# **animove**

### Temporal Patterns of Bobcat (_Lynx rufus_) Movement: Diel Activity, Activity Budgets, and Seasonal Variation Near Black Rock Forest, New York

**By**: Zhean Robby Ganituen, Jaztin Jacob Jimenez, Reece Benedict Orense, Ashley Paulianna Reyes, and Matthew Fraser Sim

---

## **introduction**

**Dataset**: LaPoint, S. 2026. Data from: Study “Carnivore movements near Black Rock Forest New York” [part]. Movebank Data Repository.

**Paper**: Oliver, R.Y. et al. 2026. Interacting effects of human presence and landscape modification on birds and mammals. Science. 392, 6800 (May 2026), 879–884. https://doi.org/10.1126/science.adq3396.

`<TODO :: add dataset description here>`

`<TODO :: add plan with the dataset>`

## **definition of terms**

We define words important to this study below:

- **Activity budget**: Is the proportion at which an animal spends their time moving versus resting.
- **Crepsucular**: Are animals that are primarily active during the twilight hours; not to be confused with _nocturnal_ animals which are active during pitch-black hours.
- **Diel Movement**: Is the synchronized, daily movement of animals moving around their habitat, occuring over the 24-hour cycle in response to light, temperature, and/or food availability.

## **research questions**

The goal of this project is to answer the general question:

> **How is the movement activity of bobcats (_Lynx rufus_) structured in time, across the daily cycle and across seasons, near Black Rock Forest, New York?**

To answer this general question, we have constructed the following research questions:

1. Is there a significant difference in the movement rate of _Lynx rufus_ during the crepuscular periods compared with midday and the middle of the night?
2. What proportion of time do _Lynx rufus_ spend in active travel versus resting, their activity budget, and how much does this vary among individuals?
3. Among the individuals whose records span multiple seasons, does the diel movement pattern change across seasons?

## **0 setup**

Let's first import the Python libraries needed for the project and define some filenames which will be used throughout the dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

# Directories
DIR_DATA = "../data/"
DIR_RAW_DATA = DIR_DATA + "raw/"
DIR_PROC_DATA = DIR_DATA + "processed/"

# Output Filenames
OUT_JOINED_DATASET = DIR_RAW_DATA + "joined.csv"
OUT_PPROC_DATASET = DIR_PROC_DATA + "preprocessed.csv"

# DataFrames global variables
MAIN_DF = ()
REF_DF = ()
JOIN_DF = ()
PPROC_DF = ()

### **0.1 dataset loading**

Now, let's load the two raw datasets as a pandas `DataFrame`.

In [ ]:
MAIN_DF = pd.read_csv(DIR_RAW_DATA + "main.csv")
REF_DF = pd.read_csv(DIR_RAW_DATA + "reference.csv")

### **0.2 joining**

Before doing any cleaning, let's first join the two. 

To get an idea of what the dataset looks like. Let's first print the columns and information from each dataframe. From this, we can come up with a methodology for joining the two.

In [ ]:
print("\t========== Main DataFrame ========== ")
print(MAIN_DF.info())

In [ ]:
print("\t========== Reference DataFrame ========== ")
print(REF_DF.info())

From here, the column `individual-local-identifier` (`ili`) from `MAIN_DF` and `animal-id` from `REF_DF` seem to be related to each other. Let's verify this claim by checking that the set of possible values for each column is the same.

In [ ]:
unique_ili_id = set(MAIN_DF["individual-local-identifier"].unique())
unique_animal_id = set(REF_DF["animal-id"].unique())

print("\t========== Is ili and animal-id a potential join key? ==========")
if unique_ili_id == unique_animal_id:
    print("YES")
    print(unique_ili_id)
else:
    print("NO")

Now that we know that `ili` and `animal-id` is the join key. We will now join `MAIN_DF` and `REF_DF` to construct `JOIN_DF`. But let's first rename `ili` to `animal-id`. Then, we'll call `pd.merge` on the join key (which is now `animal-id`).

In [ ]:
MAIN_DF = MAIN_DF.rename(columns={"individual-local-identifier": "animal-id"})
JOIN_DF = pd.merge(MAIN_DF, REF_DF, on="animal-id")

Now let's see the columns and information of the dataset after performing the join. Let's ensure that information of the two datasets were kept after merging by performing some tests.

In [ ]:
print("\t========== Joined DataFrame ========== ")
print("(Running tests...)")
print(
    "=== The total number of columns must be len(MAIN_DF.columns) + len(REF_DF.columns) - 1"
)
assert len(MAIN_DF.columns) + len(REF_DF.columns) - 1 == len(JOIN_DF.columns)

print("=== The total number of entries must be the max(len(MAIN_DF), len(REF_DF))")
assert max(len(MAIN_DF), len(REF_DF)) == len(JOIN_DF)

print()
print("(Saving joined dataset...)")
JOIN_DF.to_csv(OUT_JOINED_DATASET, index=False)

print()
print("=== JOIN_DF info")
print(JOIN_DF.info())

## **1 Dataset Preprocessing**

Now that we have the joined dataset, we can proceed with the cleaning phase.


Three source files are important for this section:
- `janitor` defines functions and a shared interface for **cleaning**.
- `transformer` defines functions and a shared interface for **transforming** data into more useful formats.
- `reducer` defines variables and functions for **dimensionality reduction** (removing unnecessary columns).

In [ ]:
import janitor
import transformer
import reducer

preprocess_step_df = JOIN_DF

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_fishers
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_bad_gps
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_with_markers
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_bad_fix
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = transformer.transform_runner(
    df=preprocess_step_df, func=transformer.add_utc_time, source_cols=["timestamp"]
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_outside_deploy
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_dup_sessions
)
report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = transformer.transform_runner(
    df=preprocess_step_df,
    func=transformer.add_local_features,
    source_cols=["timestamp-utc"],
)

report.out()

<p style="color: red;">ADD EXPLANATION.</p>

In [ ]:
preprocess_step_df, report = transformer.transform_runner(
    df=preprocess_step_df,
    func=transformer.add_movement,
    source_cols=["timestamp-utc", "location-lat", "location-long", "ground-speed"],
)

report.out()

As the final step, let's reduce the dimension of the dataset by only keeping columns important for the research questions and some other columns that can help us make sense of the dataset.

These columns are:

- <p style="color: red;">write each column name and explain why it's important.</p>


<p style="color: red;">NOTE. I added a preliminary list already. cross check if it makes sense.</p>


In [ ]:
keep_cols = [
    "event-id",
    "animal-id",
    "individual-taxon-canonical-name",
    "timestamp",
    "timestamp-utc",
    "timestamp-local",
    "hour-local",
    "date-local",
    "month-local",
    "season",
    "ground-speed",
    "heading",
    "dt-seconds",
    "step-meters",
    "derived-speed-ms",
    "location-long",
    "location-lat",
    "animal-sex",
    "animal-life-stage",
    "animal-mass",
    "deploy-on-date",
    "deploy-off-date",
    "eobs:start-timestamp",
    "eobs:horizontal-accuracy-estimate",
    "eobs:temperature",
]

In [ ]:
preprocess_step_df = reducer.reduce(df=preprocess_step_df, cols=keep_cols)

assert len(preprocess_step_df.columns) == len(keep_cols)

Now that we have performed all cleaning steps, let's store the final result in `PPROC_DF` and check its information.

In [ ]:
PPROC_DF = preprocess_step_df

print("\t========== Preprocessed DataFrame ========== ")
print(PPROC_DF.info())

PPROC_DF.to_csv(OUT_PPROC_DATASET)

# **n Exploratory Data Analysis**

<??>
What else do we add in this notebook?

The researcj questions?